# Shift-Wise Feature Engineering

This notebook rebuilds the feature matrix from raw telemetry with UTM projection, rolling-median altitude de-noising, acceleration physics, DBSCAN-based stop clustering, and mine-specific cycle segmentation.

Mine001 uses the trusted `analog_input_1` dump signal. Mine002 falls back to spatial stop clustering plus the minimum-distance cycle heuristic.

In [5]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional
import sys
import warnings

import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.cluster import DBSCAN


def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from the current working directory.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_loader import discover_data_files, split_train_test_files

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

warnings.filterwarnings("ignore")


DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "feature_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

UTM_TRANSFORMER = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

BASE_COLUMNS = [
    "vehicle",
    "mine_anon",
    "ts",
    "latitude",
    "longitude",
    "altitude",
    "speed",
    "ignition",
    "analog_input_1",
    "satellites",
    "gnss_pdop",
    "gnss_hdop",
    "disthav",
    "cumdist",
    "angle",
    "battery_level",
    "battery_current",
    "battery_voltage",
    "external_voltage",
    "axis_x",
    "axis_y",
    "axis_z",
    "gsm_signal",
    "gsm_operator",
]

TARGET_COLUMNS = ["acons", "date", "shift", "vehicle"]


@dataclass(frozen=True)
class FeatureConfig:
    utm_epsg: str = "EPSG:32645"
    shift_offset_hours: int = 2
    stop_speed_kmh: float = 1.0
    stop_min_duration_s: float = 30.0
    mine2_min_cycle_distance_m: float = 800.0
    stop_cluster_eps_m: float = 60.0
    stop_cluster_min_samples: int = 8
    stop_cluster_min_duration_s: float = 900.0
    stop_cluster_min_windows: int = 4
    stop_cluster_min_vehicles: int = 2
    altitude_window: int = 5


CONFIG = FeatureConfig()


def available_columns(path: Path) -> List[str]:
    if path.suffix.lower() == ".parquet" and pq is not None:
        return list(pq.read_schema(path).names)
    if path.suffix.lower() == ".csv":
        return list(pd.read_csv(path, nrows=0).columns)
    return []


def read_table(path: Path, columns: Iterable[str]) -> pd.DataFrame:
    requested = list(columns)
    existing = available_columns(path)
    use_cols = [c for c in requested if not existing or c in existing]
    if not use_cols:
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        return df
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, columns=use_cols)
    return pd.read_csv(path, usecols=use_cols)


def assign_shift_and_date(ts: pd.Series, offset_hours: int = 2) -> pd.DataFrame:
    ts = pd.to_datetime(ts, errors="coerce")
    hour = ts.dt.hour
    shift = np.select(
        [hour < 6, hour < 14, hour < 22],
        ["C", "A", "B"],
        default="C",
    )
    date_key = (ts + pd.Timedelta(hours=offset_hours)).dt.normalize()
    return pd.DataFrame({"date_key": date_key, "shift": shift}, index=ts.index)


def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    float_cols = [
        "latitude",
        "longitude",
        "altitude",
        "speed",
        "analog_input_1",
        "satellites",
        "gnss_pdop",
        "gnss_hdop",
        "disthav",
        "cumdist",
        "angle",
        "battery_level",
        "battery_current",
        "battery_voltage",
        "external_voltage",
        "axis_x",
        "axis_y",
        "axis_z",
        "gsm_signal",
    ]
    for col in float_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")
    if "ignition" in out.columns:
        out["ignition"] = pd.to_numeric(out["ignition"], errors="coerce").fillna(0).astype("int8")
    for col in ["vehicle", "mine_anon", "gsm_operator"]:
        if col in out.columns:
            out[col] = out[col].astype("category")
    return out


def prepare_telemetry(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ts"] = pd.to_datetime(out["ts"], errors="coerce")
    out = out.dropna(subset=["vehicle", "ts"]).copy()
    out = out.sort_values(["vehicle", "ts"]).drop_duplicates(["vehicle", "ts"], keep="last")
    out = optimize_dtypes(out)

    if "mine_anon" not in out.columns:
        out["mine_anon"] = "unknown"

    shift_frame = assign_shift_and_date(out["ts"], offset_hours=CONFIG.shift_offset_hours)
    out = pd.concat([out.reset_index(drop=True), shift_frame.reset_index(drop=True)], axis=1)
    out["vehicle"] = out["vehicle"].astype(str)
    out["mine_anon"] = out["mine_anon"].astype(str)
    out["shift"] = pd.Categorical(out["shift"], categories=["C", "A", "B"], ordered=True)
    out["veh_pos"] = out.groupby("vehicle", observed=True).cumcount()
    return out


def add_utm_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    x, y = UTM_TRANSFORMER.transform(out["longitude"].to_numpy(), out["latitude"].to_numpy())
    out["x_utm"] = x.astype("float32")
    out["y_utm"] = y.astype("float32")
    return out


def smooth_altitude(df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    out = df.copy()
    out["altitude_smooth"] = (
        out.groupby("vehicle", observed=True)["altitude"]
        .transform(lambda s: s.rolling(window=window, center=True, min_periods=1).median())
        .astype("float32")
    )
    delta_alt = out.groupby("vehicle", observed=True)["altitude_smooth"].diff().fillna(0.0)
    out["delta_alt_m"] = delta_alt.astype("float32")
    out["altitude_gain_m"] = out["delta_alt_m"].clip(lower=0).astype("float32")
    out["altitude_loss_m"] = (-out["delta_alt_m"]).clip(lower=0).astype("float32")
    return out


def compute_kinematics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values(["vehicle", "ts", "veh_pos"]).copy()

    out["dt_s"] = (
        out.groupby("vehicle", observed=True)["ts"].diff().dt.total_seconds().fillna(0.0).clip(0, 300)
    ).astype("float32")

    speed_mps = pd.to_numeric(out.get("speed"), errors="coerce").fillna(0.0) / 3.6
    out["speed_mps"] = speed_mps.astype("float32")
    delta_speed = out.groupby("vehicle", observed=True)["speed_mps"].diff().to_numpy(dtype=float)
    accel = np.zeros(len(out), dtype=np.float32)
    dt = out["dt_s"].to_numpy(dtype=float)
    valid = dt > 0
    accel[valid] = np.nan_to_num(delta_speed[valid], nan=0.0) / dt[valid]
    out["accel_mps2"] = accel
    out["hard_braking"] = (out["accel_mps2"] < -2.0).astype("int8")

    out["run_flag"] = (out["ignition"] == 1).astype("int8")
    out["idle_flag"] = ((out["ignition"] == 1) & (out["speed"] < 1.0)).astype("int8")
    out["moving_flag"] = (out["speed"] >= 1.0).astype("int8")
    out["glide_flag"] = ((out["speed"] >= 1.0) & (out["ignition"] != 1)).astype("int8")

    out["run_dt_s"] = (out["dt_s"] * out["run_flag"]).astype("float32")
    out["idle_dt_s"] = (out["dt_s"] * out["idle_flag"]).astype("float32")
    out["moving_dt_s"] = (out["dt_s"] * out["moving_flag"]).astype("float32")
    out["glide_dt_s"] = (out["dt_s"] * out["glide_flag"]).astype("float32")

    dx = out.groupby("vehicle", observed=True)["x_utm"].diff().fillna(0.0)
    dy = out.groupby("vehicle", observed=True)["y_utm"].diff().fillna(0.0)
    out["step_distance_m"] = np.hypot(dx, dy).astype("float32")
    out["cum_path_m"] = out.groupby("vehicle", observed=True)["step_distance_m"].cumsum().astype("float32")

    if {"axis_x", "axis_y", "axis_z"}.issubset(out.columns):
        axis_mag = np.sqrt(
            np.square(pd.to_numeric(out["axis_x"], errors="coerce").fillna(0.0))
            + np.square(pd.to_numeric(out["axis_y"], errors="coerce").fillna(0.0))
            + np.square(pd.to_numeric(out["axis_z"], errors="coerce").fillna(0.0))
        )
        out["axis_mag"] = axis_mag.astype("float32")
    else:
        out["axis_mag"] = 0.0

    return out


def build_stop_windows(df: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    windows: List[pd.DataFrame] = []
    for _vehicle, g in df.groupby("vehicle", observed=True, sort=False):
        g = g.sort_values("ts").copy()
        stop_flag = g["speed"].fillna(np.inf) < config.stop_speed_kmh
        if not stop_flag.any():
            continue
        seg_id = (stop_flag != stop_flag.shift(fill_value=False)).cumsum()
        g["stop_seg"] = seg_id
        stops = g.loc[stop_flag].groupby("stop_seg", observed=True).agg(
            vehicle=("vehicle", "first"),
            mine_anon=("mine_anon", "first"),
            start_ts=("ts", "first"),
            end_ts=("ts", "last"),
            date_key=("date_key", "last"),
            shift=("shift", "last"),
            start_pos=("veh_pos", "first"),
            end_pos=("veh_pos", "last"),
            duration_s=("dt_s", "sum"),
            x_median=("x_utm", "median"),
            y_median=("y_utm", "median"),
            start_cum_path_m=("cum_path_m", "first"),
            end_cum_path_m=("cum_path_m", "last"),
            mean_speed=("speed", "mean"),
            min_speed=("speed", "min"),
            ignition_share=("ignition", "mean"),
        ).reset_index(drop=True)
        stops = stops[stops["duration_s"] >= config.stop_min_duration_s].copy()
        if not stops.empty:
            windows.append(stops)

    if not windows:
        return pd.DataFrame(
            columns=[
                "vehicle",
                "mine_anon",
                "start_ts",
                "end_ts",
                "date_key",
                "shift",
                "start_pos",
                "end_pos",
                "duration_s",
                "x_median",
                "y_median",
                "start_cum_path_m",
                "end_cum_path_m",
                "mean_speed",
                "min_speed",
                "ignition_share",
                "stop_cluster",
                "cluster_duration_s",
                "cluster_windows",
                "cluster_vehicles",
                "stable_zone",
            ]
        )

    out = pd.concat(windows, ignore_index=True)
    out["stop_cluster"] = -1

    for _mine, g in out.groupby("mine_anon", observed=True):
        valid = g[["x_median", "y_median"]].notna().all(axis=1)
        if valid.sum() == 0:
            continue
        coords = g.loc[valid, ["x_median", "y_median"]].to_numpy(dtype=float)
        weights = g.loc[valid, "duration_s"].to_numpy(dtype=float)
        labels = DBSCAN(eps=config.stop_cluster_eps_m, min_samples=config.stop_cluster_min_samples).fit_predict(
            coords, sample_weight=weights
        )
        out.loc[g.index[valid], "stop_cluster"] = labels

    valid_clusters = out[out["stop_cluster"] >= 0].copy()
    if not valid_clusters.empty:
        cluster_tables = (
            valid_clusters.groupby(["mine_anon", "stop_cluster"], observed=True)
            .agg(
                cluster_duration_s=("duration_s", "sum"),
                cluster_windows=("stop_cluster", "size"),
                cluster_vehicles=("vehicle", "nunique"),
            )
            .reset_index()
        )
        out = out.merge(cluster_tables, on=["mine_anon", "stop_cluster"], how="left")
    else:
        out["cluster_duration_s"] = 0.0
        out["cluster_windows"] = 0
        out["cluster_vehicles"] = 0

    out[["cluster_duration_s", "cluster_windows", "cluster_vehicles"]] = out[
        ["cluster_duration_s", "cluster_windows", "cluster_vehicles"]
    ].fillna(0)
    out["stable_zone"] = (
        (out["stop_cluster"] >= 0)
        & (out["cluster_duration_s"] >= config.stop_cluster_min_duration_s)
        & (out["cluster_windows"] >= config.stop_cluster_min_windows)
        & (out["cluster_vehicles"] >= config.stop_cluster_min_vehicles)
    )
    return out


def build_cycle_markers(df: pd.DataFrame, stop_windows: pd.DataFrame, config: FeatureConfig) -> pd.DataFrame:
    marker_rows: List[pd.DataFrame] = []

    for vehicle, g in df.groupby("vehicle", observed=True, sort=False):
        g = g.sort_values("ts").copy()
        mine = g["mine_anon"].iloc[0] if len(g) else "unknown"

        if mine == "mine001" and "analog_input_1" in g.columns and g["analog_input_1"].notna().any():
            prev = g["analog_input_1"].shift(1)
            edge = (g["analog_input_1"] > 2.5) & ((prev <= 2.5) | prev.isna())
            markers = g.loc[edge, ["vehicle", "mine_anon", "date_key", "shift", "ts", "veh_pos", "cum_path_m", "x_utm", "y_utm"]].copy()
            if not markers.empty:
                markers["marker_source"] = "analog_dump_edge"
                markers["distance_since_last_dump_m"] = np.nan
                marker_rows.append(markers)
            continue

        if stop_windows.empty:
            continue

        candidate = stop_windows[(stop_windows["vehicle"] == vehicle) & (stop_windows["duration_s"] >= config.stop_min_duration_s)].copy()
        if candidate.empty:
            continue
        candidate = candidate.sort_values(["end_ts", "start_pos"]).copy()
        if candidate["stable_zone"].any():
            candidate = candidate[candidate["stable_zone"]].copy()
        if candidate.empty:
            continue

        selected: List[pd.Series] = []
        last_dump_cum_path_m = np.nan
        for _, row in candidate.iterrows():
            if pd.isna(last_dump_cum_path_m):
                selected.append(row)
                last_dump_cum_path_m = float(row["end_cum_path_m"])
                continue
            distance_since_last = float(row["start_cum_path_m"] - last_dump_cum_path_m)
            if distance_since_last >= config.mine2_min_cycle_distance_m:
                row = row.copy()
                row["distance_since_last_dump_m"] = distance_since_last
                selected.append(row)
                last_dump_cum_path_m = float(row["end_cum_path_m"])

        if selected:
            markers = pd.DataFrame(selected)
            markers["marker_source"] = "stop_cluster"
            if "distance_since_last_dump_m" not in markers.columns:
                markers["distance_since_last_dump_m"] = np.nan
            marker_rows.append(markers[[
                "vehicle",
                "mine_anon",
                "date_key",
                "shift",
                "start_ts",
                "end_ts",
                "start_pos",
                "end_pos",
                "duration_s",
                "x_median",
                "y_median",
                "start_cum_path_m",
                "end_cum_path_m",
                "stop_cluster",
                "cluster_duration_s",
                "cluster_windows",
                "cluster_vehicles",
                "stable_zone",
                "distance_since_last_dump_m",
                "marker_source",
            ]].rename(columns={
                "end_ts": "ts",
                "end_pos": "veh_pos",
                "end_cum_path_m": "cum_path_m",
                "x_median": "x_utm",
                "y_median": "y_utm",
            }))

    if not marker_rows:
        return pd.DataFrame(columns=["vehicle", "mine_anon", "date_key", "shift", "ts", "veh_pos", "cum_path_m", "x_utm", "y_utm", "marker_source", "distance_since_last_dump_m"])

    markers = pd.concat(marker_rows, ignore_index=True)
    markers = markers.sort_values(["vehicle", "ts", "veh_pos"]).reset_index(drop=True)
    return markers


def build_cycle_features(df: pd.DataFrame, markers: pd.DataFrame) -> pd.DataFrame:
    if markers.empty:
        return pd.DataFrame(
            columns=[
                "vehicle",
                "mine_anon",
                "date_key",
                "shift",
                "cycle_count",
                "cycle_duration_mean_s",
                "cycle_duration_median_s",
                "cycle_distance_mean_km",
                "cycle_distance_sum_km",
                "cycle_lift_mean_m",
                "cycle_lift_sum_m",
            ]
        )

    cycle_rows: List[Dict[str, object]] = []
    for vehicle, g in markers.groupby("vehicle", observed=True, sort=False):
        g = g.sort_values("veh_pos").reset_index(drop=True)
        if len(g) < 2:
            continue
        vehicle_df = df[df["vehicle"] == vehicle].sort_values("veh_pos").reset_index(drop=True)
        for idx in range(1, len(g)):
            start_marker = g.iloc[idx - 1]
            end_marker = g.iloc[idx]
            start_pos = int(start_marker["veh_pos"])
            end_pos = int(end_marker["veh_pos"])
            if end_pos <= start_pos:
                continue
            segment = vehicle_df.iloc[start_pos:end_pos + 1].copy()
            if segment.empty:
                continue
            cycle_rows.append(
                {
                    "vehicle": vehicle,
                    "mine_anon": str(end_marker["mine_anon"]),
                    "date_key": end_marker["date_key"],
                    "shift": end_marker["shift"],
                    "cycle_duration_s": float((pd.to_datetime(end_marker["ts"]) - pd.to_datetime(start_marker["ts"])).total_seconds()),
                    "cycle_distance_km": float(segment["step_distance_m"].sum() / 1000.0),
                    "cycle_lift_m": float(segment["altitude_smooth"].iloc[-1] - segment["altitude_smooth"].iloc[0]),
                    "cycle_marker_source": str(end_marker["marker_source"]),
                }
            )

    if not cycle_rows:
        return pd.DataFrame(
            columns=[
                "vehicle",
                "mine_anon",
                "date_key",
                "shift",
                "cycle_count",
                "cycle_duration_mean_s",
                "cycle_duration_median_s",
                "cycle_distance_mean_km",
                "cycle_distance_sum_km",
                "cycle_lift_mean_m",
                "cycle_lift_sum_m",
            ]
        )

    cycles = pd.DataFrame(cycle_rows)
    group_cols = ["vehicle", "mine_anon", "date_key", "shift"]
    out = cycles.groupby(group_cols, observed=True).agg(
        cycle_count=("cycle_duration_s", "size"),
        cycle_duration_mean_s=("cycle_duration_s", "mean"),
        cycle_duration_median_s=("cycle_duration_s", "median"),
        cycle_distance_mean_km=("cycle_distance_km", "mean"),
        cycle_distance_sum_km=("cycle_distance_km", "sum"),
        cycle_lift_mean_m=("cycle_lift_m", "mean"),
        cycle_lift_sum_m=("cycle_lift_m", "sum"),
    ).reset_index()
    return out


def aggregate_stop_features(stop_windows: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["vehicle", "mine_anon", "date_key", "shift"]
    if stop_windows.empty:
        return pd.DataFrame(
            columns=[
                *group_cols,
                "stop_window_count",
                "stop_duration_sum_s",
                "stop_duration_mean_s",
                "stop_duration_p90_s",
                "stable_stop_window_count",
                "stop_cluster_count",
                "dominant_stop_cluster_share",
                "stop_cluster_entropy",
            ]
        )

    base = stop_windows.groupby(group_cols, observed=True).agg(
        stop_window_count=("duration_s", "size"),
        stop_duration_sum_s=("duration_s", "sum"),
        stop_duration_mean_s=("duration_s", "mean"),
        stop_duration_p90_s=("duration_s", lambda s: float(s.quantile(0.9))),
        stable_stop_window_count=("stable_zone", "sum"),
    ).reset_index()

    valid = stop_windows[stop_windows["stop_cluster"] >= 0].copy()
    if valid.empty:
        base["stop_cluster_count"] = 0
        base["dominant_stop_cluster_share"] = 0.0
        base["stop_cluster_entropy"] = 0.0
        return base

    cluster_level = valid.groupby(group_cols + ["stop_cluster"], observed=True).agg(
        cluster_duration_s=("duration_s", "sum"),
    ).reset_index()

    cluster_summary = cluster_level.groupby(group_cols, observed=True).agg(
        stop_cluster_count=("stop_cluster", "nunique"),
        dominant_stop_cluster_share=("cluster_duration_s", lambda s: float(s.max() / (s.sum() + 1e-6))),
        stop_cluster_entropy=("cluster_duration_s", lambda s: float(-np.sum((s / (s.sum() + 1e-6)) * np.log((s / (s.sum() + 1e-6)) + 1e-9)))),
    ).reset_index()

    out = base.merge(cluster_summary, on=group_cols, how="left")
    out[["stop_cluster_count", "dominant_stop_cluster_share", "stop_cluster_entropy"]] = out[
        ["stop_cluster_count", "dominant_stop_cluster_share", "stop_cluster_entropy"]
    ].fillna(0)
    return out


def aggregate_shift_features(df: pd.DataFrame, stop_windows: pd.DataFrame, cycle_features: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["vehicle", "mine_anon", "date_key", "shift"]
    agg = df.groupby(group_cols, observed=True).agg(
        n_pings=("ts", "size"),
        active_span_s=("ts", lambda s: float((s.max() - s.min()).total_seconds()) if len(s) else 0.0),
        path_distance_km=("step_distance_m", lambda s: float(s.sum() / 1000.0)),
        total_dt_s=("dt_s", "sum"),
        run_hours=("run_dt_s", lambda s: float(s.sum() / 3600.0)),
        idle_hours=("idle_dt_s", lambda s: float(s.sum() / 3600.0)),
        moving_hours=("moving_dt_s", lambda s: float(s.sum() / 3600.0)),
        glide_hours=("glide_dt_s", lambda s: float(s.sum() / 3600.0)),
        accel_mean=("accel_mps2", "mean"),
        accel_std=("accel_mps2", "std"),
        accel_variance=("accel_mps2", lambda s: float(s.var(ddof=0))),
        hard_braking_count=("hard_braking", "sum"),
        speed_mean=("speed", "mean"),
        speed_std=("speed", "std"),
        speed_max=("speed", "max"),
        speed_p25=("speed", lambda s: float(s.quantile(0.25))),
        speed_p50=("speed", lambda s: float(s.quantile(0.50))),
        speed_p75=("speed", lambda s: float(s.quantile(0.75))),
        altitude_mean=("altitude_smooth", "mean"),
        altitude_std=("altitude_smooth", "std"),
        altitude_min=("altitude_smooth", "min"),
        altitude_max=("altitude_smooth", "max"),
        altitude_gain_m=("altitude_gain_m", "sum"),
        altitude_loss_m=("altitude_loss_m", "sum"),
        step_distance_mean_m=("step_distance_m", "mean"),
        step_distance_std_m=("step_distance_m", "std"),
        external_voltage_mean=("external_voltage", "mean"),
        external_voltage_std=("external_voltage", "std"),
        external_voltage_max=("external_voltage", "max"),
        axis_mag_mean=("axis_mag", "mean"),
        axis_mag_std=("axis_mag", "std"),
        gnss_hdop_mean=("gnss_hdop", "mean"),
        satellites_mean=("satellites", "mean"),
        cumdist_delta_km=("cumdist", lambda s: float(s.iloc[-1] - s.iloc[0]) if len(s) else 0.0),
        analog_mean=("analog_input_1", "mean"),
        analog_max=("analog_input_1", "max"),
        analog_dump_events=("analog_input_1", lambda s: float(((s > 2.5) & ((s.shift(1) <= 2.5) | s.shift(1).isna())).sum()) if s.notna().any() else 0.0),
    ).reset_index()

    x_first = df.groupby(group_cols, observed=True)["x_utm"].first().reset_index(name="x_first")
    x_last = df.groupby(group_cols, observed=True)["x_utm"].last().reset_index(name="x_last")
    y_first = df.groupby(group_cols, observed=True)["y_utm"].first().reset_index(name="y_first")
    y_last = df.groupby(group_cols, observed=True)["y_utm"].last().reset_index(name="y_last")
    disp = x_first.merge(x_last, on=group_cols).merge(y_first, on=group_cols).merge(y_last, on=group_cols)
    agg = agg.merge(disp, on=group_cols, how="left")
    agg["net_displacement_km"] = np.hypot(agg["x_last"] - agg["x_first"], agg["y_last"] - agg["y_first"]) / 1000.0
    agg = agg.drop(columns=["x_first", "x_last", "y_first", "y_last"])

    agg["coverage_hours"] = agg["active_span_s"] / 3600.0
    agg["idle_fraction"] = agg["idle_hours"] / (agg["run_hours"] + 1e-6)
    agg["moving_fraction"] = agg["moving_hours"] / (agg["coverage_hours"] + 1e-6)
    agg["climb_per_km"] = agg["altitude_gain_m"] / (agg["path_distance_km"] + 1e-6)
    agg["ext_v_x_moving_h"] = agg["external_voltage_mean"] * agg["moving_hours"]
    agg["ext_v_x_run_h"] = agg["external_voltage_mean"] * agg["run_hours"]
    agg["speed_cv"] = agg["speed_std"] / (agg["speed_mean"] + 1e-6)
    agg["altitude_range_m"] = agg["altitude_max"] - agg["altitude_min"]
    accel_abs = df.groupby(group_cols, observed=True)["accel_mps2"].apply(lambda s: float(np.mean(np.abs(s.to_numpy()))))
    agg = agg.merge(accel_abs.reset_index(name="accel_abs_mean"), on=group_cols, how="left")
    agg["hard_braking_rate"] = agg["hard_braking_count"] / (agg["n_pings"] + 1e-6)
    agg["path_speed_ratio"] = agg["path_distance_km"] / (agg["coverage_hours"] + 1e-6)

    stop_summary = aggregate_stop_features(stop_windows)
    agg = agg.merge(stop_summary, on=group_cols, how="left") if not stop_summary.empty else agg
    cycle_summary = cycle_features.copy()
    agg = agg.merge(cycle_summary, on=group_cols, how="left") if not cycle_summary.empty else agg

    fill_zero = [
        "stop_window_count",
        "stop_duration_sum_s",
        "stop_duration_mean_s",
        "stop_duration_p90_s",
        "stable_stop_window_count",
        "stop_cluster_count",
        "dominant_stop_cluster_share",
        "stop_cluster_entropy",
        "cycle_count",
        "cycle_duration_mean_s",
        "cycle_duration_median_s",
        "cycle_distance_mean_km",
        "cycle_distance_sum_km",
        "cycle_lift_mean_m",
        "cycle_lift_sum_m",
        "accel_abs_mean",
    ]
    for col in fill_zero:
        if col in agg.columns:
            agg[col] = agg[col].fillna(0)

    agg["vehicle"] = agg["vehicle"].astype(str)
    agg["mine_anon"] = agg["mine_anon"].astype(str)
    agg["shift"] = pd.Categorical(agg["shift"], categories=["C", "A", "B"], ordered=True)
    return agg


def load_summary_targets(data_dir: Path) -> pd.DataFrame:
    summary_files = sorted(data_dir.glob("smry_*_train_ordered.csv"))
    if not summary_files:
        summary_files = sorted(data_dir.glob("smry_*.csv"))
    if not summary_files:
        return pd.DataFrame(columns=["vehicle", "date_key", "shift", "acons"])

    summary = pd.concat([pd.read_csv(path) for path in summary_files], ignore_index=True)
    if "date" in summary.columns:
        summary["date_key"] = pd.to_datetime(summary["date"], errors="coerce").dt.normalize()
    elif "date_dpr" in summary.columns:
        summary["date_key"] = pd.to_datetime(summary["date_dpr"], errors="coerce").dt.normalize()
    else:
        summary["date_key"] = pd.NaT
    summary["shift"] = summary["shift"].astype(str).str.upper()
    summary["vehicle"] = summary["vehicle"].astype(str)
    summary["acons"] = pd.to_numeric(summary["acons"], errors="coerce")
    before = len(summary)
    summary = summary[summary["acons"].notna() & (summary["acons"] > 0)].copy()
    dropped = before - len(summary)
    print(f"[targets] Loaded {before} rows, dropped {dropped} nonpositive/invalid target rows.")
    return summary[["vehicle", "date_key", "shift", "acons"]].drop_duplicates(["vehicle", "date_key", "shift"], keep="last")


def merge_targets(features: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    if targets.empty:
        return features.copy()
    out = features.merge(targets, on=["vehicle", "date_key", "shift"], how="left")
    labeled = out["acons"].notna()
    print(f"[targets] Matched labels for {labeled.sum()} / {len(out)} feature rows.")
    return out


def build_feature_matrix(file_paths: List[Path], targets: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    frames: List[pd.DataFrame] = []
    required = sorted(set(BASE_COLUMNS))

    for idx, path in enumerate(file_paths, start=1):
        print(f"[features] {idx:02d}/{len(file_paths):02d} {path.name}")
        df = read_table(path, required)
        df = prepare_telemetry(df)
        df = add_utm_coordinates(df)
        df = smooth_altitude(df, window=CONFIG.altitude_window)
        df = compute_kinematics(df)

        stop_windows = build_stop_windows(df, CONFIG)
        cycle_markers = build_cycle_markers(df, stop_windows, CONFIG)
        cycle_features = build_cycle_features(df, cycle_markers)
        shift_features = aggregate_shift_features(df, stop_windows, cycle_features)
        frames.append(shift_features)

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not out.empty:
        out = out.sort_values(["vehicle", "date_key", "shift"]).drop_duplicates(["vehicle", "date_key", "shift"], keep="last")
    if targets is not None and not targets.empty:
        out = merge_targets(out, targets)
    return out


def feature_columns(df: pd.DataFrame) -> List[str]:
    excluded = {
        "vehicle",
        "mine_anon",
        "date_key",
        "shift",
        "acons",
        "start_ts",
        "end_ts",
    }
    cols: List[str] = []
    for col in df.columns:
        if col in excluded:
            continue
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if df[col].dtype == "object":
            continue
        cols.append(col)
    return cols

In [6]:
files = discover_data_files(DATA_DIR)
train_files, test_files = split_train_test_files(files.telemetry_files)
print(f"[setup] train files: {len(train_files)} | test files: {len(test_files)}")

if not train_files or not test_files:
    print("[setup] No telemetry files were found under DATA_DIR in this workspace.")
    print("[setup] Mount or extract the competition data into data/ before rerunning this cell.")
else:
    summary_targets = load_summary_targets(DATA_DIR)
    train_features = build_feature_matrix(train_files, targets=summary_targets)
    test_features = build_feature_matrix(test_files, targets=None)

    feature_cols = [c for c in feature_columns(train_features) if c in test_features.columns]
    train_model = train_features[["vehicle", "mine_anon", "date_key", "shift", "acons", *feature_cols]].copy()
    test_model = test_features[["vehicle", "mine_anon", "date_key", "shift", *feature_cols]].copy()

    train_path = CACHE_DIR / "train_shift_features.parquet"
    test_path = CACHE_DIR / "test_shift_features.parquet"
    train_model.to_parquet(train_path, index=False)
    test_model.to_parquet(test_path, index=False)

    print(f"[output] train_model shape: {train_model.shape}")
    print(f"[output] test_model shape:  {test_model.shape}")
    print(f"[output] saved -> {train_path}")
    print(f"[output] saved -> {test_path}")
    print("[output] target summary:")
    print(train_model["acons"].describe())
    print("[output] top missing rates:")
    print(train_model.isna().mean().sort_values(ascending=False).head(20))

[setup] train files: 0 | test files: 0
[setup] No telemetry files were found under DATA_DIR in this workspace.
[setup] Mount or extract the competition data into data/ before rerunning this cell.
